## scGPT-mini

### IMPORTS AND GLOBAL CONSTANTS

In [1]:
import torch
import datetime
import json
import scanpy as sc
import transformers
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Dataset, DataChunk

In [2]:
EMBEDDING_SIZE = 12
#INPUT_SIZE = 400 # HIGH VARIABLE GENE COUNT
NUM_HEADS = 3
GENE_VOCAB_SIZE = 1200 #len(gene_vocab)
NUM_BINS = 20
CONDITION_VOCAB_SIZE = 0 # overwritten by len(condition_vocab)

### TOKENIZERS

In [3]:
with open("data/gene-id-vocab.json") as f:
    gene_vocab = json.load(f)
def tokenize_gene(gene_name):
    return gene_vocab[gene_name]

with open("data/condition-vocab.json") as f:
    condition_vocab = json.load(f)
def tokenize_condition(condition_name):
    return condition_vocab[condition_name]
CONDITION_VOCAB_SIZE = len(condition_vocab)

### PREPROCESSING

In [4]:
a = torch.tensor([1,2,3,0,24,5,6,4,0,3,0,0,2,11,45], dtype=torch.float)
z = a == 0
nz = a[~z]
print(f"nz: {nz}")
l = torch.linspace(0, 1, 8)
print(f"l: {l}")
edges = torch.quantile(nz, l)
print(f"edges: {edges}")
bins = torch.bucketize(nz, edges)
print(f"bins: {bins}")

nz: tensor([ 1.,  2.,  3., 24.,  5.,  6.,  4.,  3.,  2., 11., 45.])
l: tensor([0.0000, 0.1429, 0.2857, 0.4286, 0.5714, 0.7143, 0.8571, 1.0000])
edges: tensor([ 1.0000,  2.0000,  2.8571,  3.2857,  4.7143,  6.7143, 18.4286, 45.0000])
bins: tensor([0, 1, 3, 7, 5, 5, 4, 3, 1, 6, 7])


In [5]:
class Preprocessor:
    """
        Transformation of data for efficient optimization in this order:
        - A log transform to scale data
        - Binning for statistical robustness of prediction in cases of noise
        - Masking for selecting labels
    """
    def __init__(self, log1p: bool = True, binning: bool = True, binning_strategy: str = "quantile", 
                 num_bins: int = NUM_BINS, mask_prob: float = 0.15, mask_token: str = "<MASK>"):
        
        self.log1p = log1p
        
        self.binning = binning
        self.binning_strategy = binning_strategy
        self.num_bins = num_bins
        
        self.mask_prob = mask_prob
        self.mask_token = mask_token
        self.mask_bin = num_bins #+ 1


    class Cell:
        pass
        
    def cell_binner(self, cells):
        # LOG TRANSFORM
        if self.log1p:
            cells = torch.log1p(cells)

        # BINNING STRATEGY
        if self.binning_strategy == "quantile":
            
            q = torch.linspace(0, 1, steps=self.num_bins+1)
            edges = torch.quantile(cells, q)
            binned_cells = torch.bucketize(cells, edges)
        elif self.binning_strategy == "linear":
            pass

        # 
        return binned_cells

    
    def cell_masker(self, binned_cells):
        cells = binned_cells
        
        labels = cells.clone()
        masked_cells = cells.clone()
        
        mask = torch.rand(cells.shape) < self.mask_prob
        nonzeros = masked_cells > 0
        
        masked_cells[mask] = self.mask_bin
        masked_cells[nonzeros] += 1   # SEPARATE ZEROS AS A SEPARATE BIN ENTIRELY

        #masked_gene_ids = gene_ids[mask]
        
        return ( torch.tensor(binned_cells, dtype=torch.long), 
                 torch.tensor(masked_cells, dtype=torch.long), 
                 torch.tensor(labels, dtype=torch.long), torch.tensor(mask, dtype=torch.bool)
               )

        
    def __call__(self, cells):
        
        # LOG TRANSFORMATION
        if self.log1p:
            cells = torch.log1p(cells)
        
        # BINNING (LINEAR, QUANTILE)
        if self.binning:
            binned_cells = self.cell_binner(cells)
        
        # MASKING
        binned_cells, masked_cells, labels, masks = self.cell_masker(cells)

            
        return binned_cells, masked_cells, labels, masks

### ARCHITECTURE

In [6]:
class InputLayer(nn.Module):

    def __init__(self):
        super().__init__()
        self.emb_g = nn.Embedding(num_embeddings=GENE_VOCAB_SIZE, embedding_dim=EMBEDDING_SIZE)
        self.emb_x = nn.Embedding(num_embeddings=NUM_BINS+2, embedding_dim=EMBEDDING_SIZE)
        #self.emb_c = nn.Embedding(num_embeddings=INPUT_SIZE, embedding_dim=EMBEDDING_SIZE)
        
    def forward(self, gene_ids, expr_bins, condition_ids=None):        
        
        emb = self.emb_g(gene_ids) + self.emb_x(expr_bins) #+ self.emb_c(x[2])
        
        return emb

        
class AttentionLayer(nn.Module):
        
    def __init__(self, num_heads=NUM_HEADS):
        super().__init__()
        self.Wq = nn.Linear(EMBEDDING_SIZE, EMBEDDING_SIZE, bias=False)
        self.Wk = nn.Linear(EMBEDDING_SIZE, EMBEDDING_SIZE, bias=False)
        self.Wv = nn.Linear(EMBEDDING_SIZE, EMBEDDING_SIZE, bias=False)
        self.Wo = nn.Linear(EMBEDDING_SIZE, EMBEDDING_SIZE)

        self.num_heads = num_heads
        self.head_dim = EMBEDDING_SIZE // self.num_heads
    

    def forward(self, x):
        B, T, D = x.shape
        
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        # ( B, T, D)
        
        Q = Q.view(B, T, self.num_heads, self.head_dim)
        K = K.view(B, T, self.num_heads, self.head_dim)
        V = V.view(B, T, self.num_heads, self.head_dim)
        # (B, T, H, Dh)
        
        Q = Q.transpose(1,2)
        K = K.transpose(1,2)
        V = V.transpose(1,2)
        # (B, H, T, Dh)
        
        scores = Q @ K.transpose(-2, -1)
        # (B, H, T, T)
        
        dk = np.sqrt(EMBEDDING_SIZE)
        attention = torch.softmax( scores / dk , dim=-1)@V 
        # (B, H, T, Dh)

        attention = attention.transpose(1, 2)
        # (B, T, H, Dh)

        attention = attention.reshape(B, T, EMBEDDING_SIZE)
        #attention = attention.contiguous().view(B, T, EMBEDDING_SIZE)
        
        return attention


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin1 = nn.Linear(EMBEDDING_SIZE, 4*EMBEDDING_SIZE)
        self.relu1 = nn.ReLU()
        self.lin2 = nn.Linear(4*EMBEDDING_SIZE, EMBEDDING_SIZE)

    def forward(self, x):
        x = self.lin1(x)
        x = self.relu1(x)
        x = self.lin2(x)
        
        return x

class AttentionBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention_layer = AttentionLayer()
        self.mlp = MLP()
        self.ln = nn.LayerNorm(EMBEDDING_SIZE)

    def forward(self, x):
        residual1 = x
        
        # LAYER NORM
        x = self.ln(x)
        
        # ATTENTION LAYER
        x = self.attention_layer(x)

        # RESIDUAL
        x = x + residual1

        residual2 = x
        # LAYER NORM
        x = self.ln(x)
        
        # MLP
        x = self.mlp(x)

        # RESIDUAL
        x = x + residual2

        return x


class scGPTModel(nn.Module):
    def __init__(self, num_attention_layers, num_bins=NUM_BINS):
        super().__init__()
        self.input_layer = InputLayer()
        self.ln1 = nn.LayerNorm(EMBEDDING_SIZE)
        self.attention_blocks = nn.ModuleList([
            AttentionBlock()
            for _ in range(num_attention_layers)
        ])
        self.l1 = nn.Linear(
            EMBEDDING_SIZE,
            4*EMBEDDING_SIZE
        )
        self.relu1 = nn.ReLU()
        self.l2 = nn.Linear(
            4*EMBEDDING_SIZE,
            num_bins+1 # RESERVED ZERO-EXPRESSION
        )
        

    def forward(self, gene_ids, expr_bins):
        
        # INPUT LAYER
        x = self.input_layer(gene_ids, expr_bins)
        
        # ATTENTION BLOCKS
        for attention_block in self.attention_blocks:
            x = attention_block(x)
            
        x = self.l1(x)
        x = self.relu1(x)
        x = self.l2(x)

        return x

### DATASET

In [7]:
adata = sc.read_10x_mtx(
    "data/filtered_feature_bc_matrix",
    var_names="gene_symbols",
    cache=True,
)
adata

AnnData object with n_obs × n_vars = 11769 × 33538
    var: 'gene_ids', 'feature_types'

In [8]:
#adata = sc.datasets.pbmc3k_processed()

class CellDataset(Dataset):

    def __init__(self, gene_ids, cells, preprocessor=None):
        self.gene_ids = gene_ids.long()
        self.cells = cells.long()
        self.preprocessor = preprocessor or Preprocessor()

    def preprocess(self):
        #cells = self.cells[idx]
        self.binned_cells, self.masked_cells, self.labels, self.masks = self.preprocessor(self.cells)
        
    def __len__(self):
        return self.cells.shape[0]
        

    def __getitem__(self, idx):
        
        return self.binned_cells[idx], self.masked_cells[idx], self.labels[idx], self.masks[idx]
        
    __call__ = preprocess

### TRAINING

In [9]:
gene_ids = torch.as_tensor(
    [ tokenize_gene(gene) for gene in list(adata.var['gene_ids'])[:GENE_VOCAB_SIZE]], dtype=torch.long
)

train_cells = torch.tensor(
    adata.X.toarray()[300:,:GENE_VOCAB_SIZE],
    dtype=torch.long
)
train_dataset = CellDataset(
    gene_ids,
    train_cells
)
train_dataset()
train_loader = DataLoader(
    train_dataset,
    batch_size=3,
    shuffle=True
)

/tmp/ipykernel_112997/3446817467.py:35: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at /pytorch/aten/src/ATen/native/BucketizationUtils.h:32.)
  binned_cells = torch.bucketize(cells, edges)
/tmp/ipykernel_112997/3446817467.py:57: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return ( torch.tensor(binned_cells, dtype=torch.long),
/tmp/ipykernel_112997/3446817467.py:58: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(maske

In [10]:
gene_ids = train_dataset.gene_ids
epochs = 7
model = scGPTModel(num_attention_layers=2)
model.train()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=model.parameters(), lr=1e-4)
for epoch in range(epochs):
    
    total_loss = 0
    batch_ = 1
    for binned_cells, masked_cells, labels, masks in train_loader:
        #print(binned_cells.max(), masked_cells.max(), labels.max())
        #print("labels", labels.shape, "labels[masks]", labels[masks].shape) 
        # ----------------------------------------
        # Forward
        # ----------------------------------------
        #gene_ids = gene_ids[masks]
        logits = model(gene_ids, masked_cells)#?
          
        #print("logits", logits.shape, "masks", masks.shape, "logits[masks]", logits[masks].shape) 
        # logits:
        # (batch, seq_len, num_bins)

        # ----------------------------------------
        # Keep only masked positions
        # ----------------------------------------
        
        
        logits = logits[masks]
        labels = labels[masks]
        
        # logits:
        # (N_masked, num_bins)

        # labels:
        # (N_masked)

        # ----------------------------------------
        # Loss
        # ----------------------------------------
        #logits = logits.permute(0, 3, 1, 2) # TO FULFILL CROSSENTROPYLOSS CLASSES POSITION REQUIREMENT
        loss = criterion(logits, labels)

        # ----------------------------------------
        # Backprop
        # ----------------------------------------

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        batch_ += 1

    
    print(
        f"Epoch {epoch+1}: "
        f"{total_loss/len(train_loader):.6f}"
    )

Epoch 1: 0.225681
Epoch 2: 0.057187
Epoch 3: 0.050427
Epoch 4: 0.047311
Epoch 5: 0.045496
Epoch 6: 0.044410
Epoch 7: 0.043811
Epoch 8: 0.043403
Epoch 9: 0.043133
Epoch 10: 0.042925
Epoch 11: 0.042745
Epoch 12: 0.042630
Epoch 13: 0.042501
Epoch 14: 0.042423
Epoch 15: 0.042338


### SAVE MODEL

In [11]:
def save_model():
    
    model_state_dict = model.state_dict()
    torch.save(model_state_dict, f"data/checkpoints/scgpt-mini-{datetime.date.today()}.pt")
    
save_model()
saved_model = torch.load(f"data/checkpoints/scgpt-mini-{datetime.date.today()}.pt", map_location="cpu")

### EMBEDDING EXTRACTION

In [12]:
def embedding_from_gene_id(gene_id: str="ENSG00000243485"): 
    """
        e.g "ENSG00000243485"
    """
    emb_g = model.get_parameter("input_layer.emb_g.weight")
    
    index = tokenize_gene(gene_id)
    a = torch.zeros(GENE_VOCAB_SIZE)#.detach()
    a[index] = 1
    
    return a@emb_g.detach()#.numpy()
    
def embedding_from_gene_index(gene_index):
    emb_g = model.get_parameter("input_layer.emb_g.weight")
    
    a = torch.zeros(GENE_VOCAB_SIZE)
    a[gene_index] = 1
    return (a@emb_g).detach()#.numpy()

In [13]:
embedding_from_gene_id("ENSG00000243485")

tensor([-0.6779,  2.4581, -0.2265, -0.0276,  1.4784, -0.3612,  0.0969, -0.3302,
         0.7857, -1.3396,  0.0036, -1.7052])

In [14]:
embedding_from_gene_id("ENSG00000237613")

tensor([ 0.7846, -0.8339,  0.1678, -0.6319, -0.0984,  0.0986,  0.8887,  0.6311,
         0.8415,  0.0412, -0.2068, -0.3633])

In [15]:
def update_embeddings_json_file():
    emb_dict = {}
    with open("data/embeddings.json", "w") as f:
        for gene_id, index in list(gene_vocab.items())[:GENE_VOCAB_SIZE]:
            emb = embedding_from_gene_index(index)
            emb_dict[gene_id] = str(emb)
    
        json.dump(emb_dict, f)
update_embeddings_json_file()

In [16]:
def embedding_from_json_file(gene_id, filename="embeddings.json"):
    from torch import tensor
    with open("data/embeddings.json") as f:
        emb_dict = json.load(f)
        emb_str = emb_dict[gene_id]
        emb = eval(emb_str).detach().clone()

        return emb
    

In [17]:
embedding_from_json_file("ENSG00000243485")

tensor([-0.6779,  2.4581, -0.2265, -0.0276,  1.4784, -0.3612,  0.0969, -0.3302,
         0.7857, -1.3396,  0.0036, -1.7052])

### INFERENCE

In [18]:
model.eval()

gene_ids = torch.tensor([ tokenize_gene(gene) for gene in list(adata.var['gene_ids'])[:GENE_VOCAB_SIZE]])

inf_cells = torch.tensor(
    adata.X.toarray()[:300,:GENE_VOCAB_SIZE],
    dtype=torch.long
)
inf_dataset = CellDataset(
    gene_ids,
    inf_cells
)
inf_dataset.preprocess()
inf_loader = DataLoader(
    inf_dataset,
    batch_size=3,
    shuffle=True
)

/tmp/ipykernel_112997/3446817467.py:57: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return ( torch.tensor(binned_cells, dtype=torch.long),
/tmp/ipykernel_112997/3446817467.py:58: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(masked_cells, dtype=torch.long),
/tmp/ipykernel_112997/3446817467.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(labels, dtype=torch.long), torch.tensor(mask, dtype=torch.bool)


In [19]:
accs = []
for binned_cells, masked_cells, labels, mask  in inf_loader:
    logits = model(gene_ids, masked_cells)
    #print(f"{binned_cells.shape} - {masked_cells.shape} - {labels.shape} - {mask.shape}")
    #print("mask : ", mask)
    
    pred = logits.argmax(dim=-1)
    acc = (pred[mask] == labels[mask]).sum() / pred[mask].shape[0]
    accs.append(acc)
    #print(acc)
    if sum(pred[mask] * labels[mask]) > 0:
        #print("good pred : ", pred[mask],"\n\n", labels[mask], "\n\n\n")
        #print(pred[mask])
        nz_pred = torch.nonzero(pred[mask])
        nz_lab = torch.nonzero(labels[mask])
        print(torch.isin(nz_pred, nz_lab).sum())
        #print(acc)
        
    else:
        pass
        #print((pred[mask] == labels[mask]).sum() / pred[mask].shape[0])
        #print("bad pred : ", pred[mask],"\n\n\n")
print("Mean accuracy", torch.tensor(accs, dtype=torch.float).mean())

tensor(5)
tensor(1)
tensor(3)
tensor(3)
tensor(10)
tensor(7)
tensor(11)
tensor(5)
tensor(7)
tensor(9)
tensor(3)
tensor(16)
tensor(18)
tensor(2)
tensor(7)
tensor(9)
tensor(3)
tensor(4)
tensor(9)
tensor(4)
tensor(11)
tensor(4)
tensor(25)
tensor(5)
tensor(10)
tensor(9)
tensor(15)
tensor(2)
tensor(14)
tensor(8)
tensor(6)
tensor(14)
tensor(19)
tensor(10)
tensor(8)
tensor(5)
tensor(7)
tensor(4)
tensor(17)
tensor(4)
tensor(6)
tensor(7)
tensor(17)
tensor(9)
tensor(4)
tensor(4)
tensor(5)
tensor(9)
tensor(6)
tensor(6)
tensor(2)
tensor(8)
tensor(13)
tensor(3)
tensor(4)
tensor(8)
tensor(9)
tensor(11)
tensor(5)
tensor(9)
tensor(6)
tensor(9)
tensor(16)
tensor(6)
tensor(7)
tensor(5)
tensor(19)
tensor(10)
tensor(3)
tensor(21)
tensor(9)
tensor(17)
tensor(7)
tensor(4)
tensor(16)
tensor(1)
tensor(11)
tensor(9)
tensor(11)
tensor(4)
tensor(15)
tensor(7)
tensor(8)
tensor(5)
tensor(3)
tensor(13)
tensor(7)
tensor(4)
tensor(2)
tensor(7)
tensor(5)
tensor(10)
tensor(8)
tensor(15)
tensor(7)
tensor(12)
tensor(13)


In [20]:
def reveal_most_expressed_genes(dataset, gene_vocab=gene_vocab):
    gene_ids, expr_bins, masked_bins, labels, mask = dataset
    logits = model(gene_ids, expr_bins)
    pass

### EVAL

In [21]:
#model.eval()



In [22]:
mask = torch.tensor([False, False, True, True])
a = torch.tensor([ 4,6,8,4])
a.nonzero()
torch.nonzero(a)

tensor([[0],
        [1],
        [2],
        [3]])

In [23]:
adata.X

<Compressed Sparse Column sparse matrix of dtype 'float32'
	with 24825783 stored elements and shape (11769, 33538)>